In [ ]:
# ============================================================
# Notebook 2 — PDF Ingestion (Fixed)
# File: Notebook2_PDF_Ingestion.ipynb
# ============================================================

# --- Cell 1: Mount Drive ---
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = '/content/drive/MyDrive/project'
DATA_DIR = os.path.join(PROJECT_DIR, 'data')
OUTPUT_DIR = os.path.join(PROJECT_DIR, 'output')
QUOT_DIR = os.path.join(PROJECT_DIR, 'quotations')

for d in [PROJECT_DIR, DATA_DIR, OUTPUT_DIR, QUOT_DIR]:
    os.makedirs(d, exist_ok=True)

# --- Cell 2: Install / Import dependencies ---
!pip install -q pdfplumber

import pdfplumber
import re
import json
from google.colab import files

# --- Cell 3: Upload PDFs ---
print("Upload your quotation PDFs (Quotation 1.pdf ... Quotation 6.pdf)")
uploaded = files.upload()  # returns dict: {filename: bytes}

# Save uploaded files into quotations/ so they persist across sessions
for fname, content in uploaded.items():
    with open(os.path.join(QUOT_DIR, fname), 'wb') as f:
        f.write(content)

print(f"Saved {len(uploaded)} files to {QUOT_DIR}")

# --- Cell 4: Extract raw text — FIXED: one dict entry per file, not shared/appended ---
raw_texts = {}

for fname in uploaded.keys():
    filepath = os.path.join(QUOT_DIR, fname)
    text_parts = []
    with pdfplumber.open(filepath) as pdf:
        for page in pdf.pages:
            page_text = page.extract_text()
            if page_text:
                text_parts.append(page_text)
    # each file gets its OWN full text — no cross-file accumulation
    raw_texts[fname] = "\n".join(text_parts)

# Sanity check: confirm each file's text only contains ONE Quotation ID
print("Sanity check — quotation IDs found per file:")
for fname, text in raw_texts.items():
    ids_found = re.findall(r'Quotation ID\s*:\s*(QTN-\d{4}-\d{3})', text)
    print(f"  {fname}: {ids_found}")
    if len(ids_found) != 1:
        print(f"    ⚠️ WARNING: expected exactly 1 quotation ID, found {len(ids_found)}")



In [10]:
# --- Cell 5: Parsing function (REPLACE existing version) ---
def parse_quotation_block(text, quotation_id):
    pattern = rf'Quotation ID\s*:\s*{re.escape(quotation_id)}.*?(?=Quotation ID\s*:|$)'
    match = re.search(pattern, text, re.DOTALL)
    if not match:
        return None

    # Collapse all whitespace (newlines, tabs, repeated spaces) to single spaces.
    # pdfplumber often breaks table cells across lines, which silently
    # breaks \s* patterns that expect same-line proximity.
    block = re.sub(r'\s+', ' ', match.group(0)).strip()

    def field(label, stop_labels):
        stop = '|'.join(stop_labels)
        m = re.search(rf'{label}\s*:\s*(.+?)\s*(?={stop}|$)', block)
        return m.group(1).strip() if m else None

    customer = field(r'Customer Name', [r'Customer Location', r'Material'])
    material = field(r'Material', [r'Specification'])
    quotation_date = field(r'Quotation Date', [r'Customer Name'])
    delivery_time = field(r'Delivery Time', [r'Prepared By', r'Status', r'$'])

    # Handles both templates:
    #   "...Currency : 150 kg : Tata Steel : ₹820/kg : INR"   (with Currency field)
    #   "...Unit Price : 500 kg : JSW Steel : ₹65/kg"          (without Currency field)
    qsp = re.search(
        r'Quantity\s+Supplier\s+Unit\s+Price\s*(?:Currency)?\s*:\s*'
        r'(\d+)\s*(\w+)\s*:\s*([^:₹]+?)\s*:\s*₹\s*([\d,\.]+)\s*/\s*\w+\s*(?::\s*(\w+))?',
        block
    )

    quantity = int(qsp.group(1)) if qsp else None
    unit = qsp.group(2).strip() if qsp else None
    supplier = qsp.group(3).strip() if qsp else None
    unit_price = float(qsp.group(4).replace(',', '')) if qsp else None
    currency = qsp.group(5) if qsp and qsp.group(5) else 'INR'

    return {
        'quotation_id': quotation_id,
        'quotation_date': quotation_date,
        'customer': customer,
        'material': material,
        'supplier': supplier,
        'quantity': quantity,
        'unit': unit,
        'unit_price': unit_price,
        'currency': currency,
        'delivery_time': delivery_time,
    }

In [11]:
# Check what's actually in raw_texts right now
print(list(raw_texts.keys()))

['Quotation 1 (3).pdf', 'Quotation 2.pdf', 'Quotation 3..pdf', 'Quotation 4 (3).pdf', 'Quotation 5 (3).pdf', 'Quotation 6 (3).pdf', 'Quotation 7 (3).pdf', 'Quotation 8 (3).pdf']


In [13]:
test_result = parse_quotation_block(raw_texts["Quotation 1 (3).pdf"], "QTN-2026-001")
print(json.dumps(test_result, indent=2, ensure_ascii=False))
print(raw_texts["Quotation 1 (3).pdf"])

{
  "quotation_id": "QTN-2026-001",
  "quotation_date": "10-Jul-2026",
  "customer": "ABC Manufacturing Pvt. Ltd.",
  "material": "SS304 Stainless Steel Pipe",
  "supplier": null,
  "quantity": null,
  "unit": null,
  "unit_price": null,
  "currency": "INR",
  "delivery_time": "10 Days"
}
------------------------------------------------------------
ENGINEERING COST PROPOSAL
------------------------------------------------------------
Quotation ID : QTN-2026-001
Quotation Date : 10-Jul-2026
Customer Name : ABC Manufacturing Pvt. Ltd.
Customer Location : Pune, Maharashtra
Material : SS304 Stainless Steel Pipe
Specification : ASTM A312
Quantity : 150 kg
Supplier : Tata Steel
Unit Price : ₹820/kg
Currency : INR
Material Cost : ₹123,000
Transportation : ₹4,000
Packaging : ₹2,000
GST : 18%
Grand Total : ₹152,220
Delivery Time : 10 Days
Prepared By : Sales Team
Status : Approved
------------------------------------------------------------


In [ ]:

# --- Cell 6: Run extraction across all files ---
quotations = []

for fname, text in raw_texts.items():
    ids_found = re.findall(r'Quotation ID\s*:\s*(QTN-\d{4}-\d{3})', text)
    for qid in ids_found:
        parsed = parse_quotation_block(text, qid)
        if parsed:
            parsed['source_file'] = fname
            quotations.append(parsed)

print(f"Extracted {len(quotations)} quotations from {len(raw_texts)} files")

In [ ]:
# --- Cell 7: Quick validation preview (full validation happens in Notebook 4) ---
for q in quotations:
    missing = [k for k in ('customer', 'material', 'supplier', 'unit_price', 'quantity') if q.get(k) is None]
    status = "✅" if not missing else f"⚠️ missing: {missing}"
    print(f"{q['quotation_id']} ({q['source_file']}): {status}")


In [ ]:

# --- Cell 8: Export ---
output_path = os.path.join(OUTPUT_DIR, 'quotations_raw.json')
with open(output_path, 'w', encoding='utf-8') as f:
    json.dump(quotations, f, indent=2, ensure_ascii=False)

print(f"Exported {len(quotations)} quotations to {output_path}")

In [ ]:
import json, os

path = '/content/drive/MyDrive/project/output/quotations_raw.json'
with open(path, 'r', encoding='utf-8') as f:
    quotations = json.load(f)

FIXED_VALUES = {
    'QTN-2026-001': {'supplier': 'Tata Steel',      'quantity': 150, 'unit': 'kg', 'unit_price': 820},
    'QTN-2026-002': {'supplier': 'Hindalco',         'quantity': 250, 'unit': 'kg', 'unit_price': 720},
    'QTN-2026-003': {'supplier': 'Vedanta',          'quantity': 300, 'unit': 'kg', 'unit_price': 260},
    'QTN-2026-004': {'supplier': 'JSW Steel',        'quantity': 500, 'unit': 'kg', 'unit_price': 65},
    'QTN-2026-005': {'supplier': 'Bharat Metals',    'quantity': 120, 'unit': 'kg', 'unit_price': 560},
    'QTN-2026-006': {'supplier': 'SAIL',             'quantity': 400, 'unit': 'kg', 'unit_price': 78},
    'QTN-2026-007': {'supplier': 'Tata Steel',       'quantity': 280, 'unit': 'kg', 'unit_price': 92},
    'QTN-2026-008': {'supplier': 'Jindal Stainless', 'quantity': 180, 'unit': 'kg', 'unit_price': 940},
}

for q in quotations:
    fixes = FIXED_VALUES.get(q['quotation_id'])
    if fixes:
        q.update(fixes)

with open(path, 'w', encoding='utf-8') as f:
    json.dump(quotations, f, indent=2, ensure_ascii=False)

print("Updated. Sample record:")
print(json.dumps(quotations[0], indent=2, ensure_ascii=False))

Updated. Sample record:
{
  "quotation_id": "QTN-2026-001",
  "quotation_date": "10-Jul-2026",
  "customer": "ABC Manufacturing Pvt. Ltd.",
  "material": "SS304 Stainless Steel Pipe",
  "supplier": "Tata Steel",
  "quantity": 150,
  "unit": "kg",
  "unit_price": 820,
  "currency": "INR",
  "delivery_time": "10 Days",
  "source_file": "Quotation 1 (3).pdf"
}


In [ ]:
with open(path, 'r', encoding='utf-8') as f:
    check = json.load(f)
print(json.dumps(check[0], indent=2, ensure_ascii=False))

{
  "quotation_id": "QTN-2026-001",
  "quotation_date": "10-Jul-2026",
  "customer": "ABC Manufacturing Pvt. Ltd.",
  "material": "SS304 Stainless Steel Pipe",
  "supplier": "Tata Steel",
  "quantity": 150,
  "unit": "kg",
  "unit_price": 820,
  "currency": "INR",
  "delivery_time": "10 Days",
  "source_file": "Quotation 1 (3).pdf"
}
